# Business Understanding

## Problem Statement
The goal of this project is to develop a machine learning model capable of identifying the smallest set of features that can reliably predict whether an early-stage startup will succeed or fail. Additionally, the project will explore how combinations of factors influence the startup's growth and long-term performance.

## Business Objectives

### Background
Few startups rapidly scale into successful businesses, while many others fail due to factors like poor market fit, insufficient funding, lack of product traction, etc. Investors, founders, and analysts often rely on typical benchmarks, historical records of success, and experience to evaluate a startup's potential. However, these methods may overlook some complex relationships hidden within the data.

This project will explore whether machine learning can be used to identify the most common and key patterns associated with startup success or failure.

The dataset contains startup-related information such as funding rounds, team size, market potential, revenue, investory type, and more.

### Objectives
The primary business objective is to use the above factors in building predictive models that estimate the likelihood of success.

Secondary objectives include:
* Identifying which startup features have the strongest influence on success
* Understand how feature combinations influce startup outcomes
* Provide investors and analysts with a more data-driven approach to evaluating the potential of a startup



## Business Success Criteria

The project will be considered successful if the final machine learning model successfully predicts outcomes with reasonable reliability.

Additional success criteria include:
* The development of multiple models to compare and determine which approach performs best
* Producing interpretable results identifying which features contributed most to the prediction
* Demonstrating that machine learning can evaluate startups and support decision-making processes for investors, founders, and analysts

## Situational & Data Assessment

### Resources
The following resources will be used in this project:
* Business and financial information from startups
* Python programming language
* Jupyter Notebook dev environment
* Data science libraries including pandas, NumPy, matplotlib, seaborn, and scikit-learn
* ML algorithms and models such as Logistic Regression, Regularization, and Decision Trees

### Assumptions & Constraints
#### Assumptions:
1. The dataset accurately represents real startup business data
2. The labeled outcomes correctly identify success or failure
3. The dataset contains enough information to support early-stage startup predictions

#### Constraints
1. The project is currently limited to the data provided in the dataset. Additional sourced data is out of scope.
2. Computational limitations may reduce the ability to test highly complex models
3. There may be a large number of missing or unusable values in the dataset

### Risks and Contingencies
1. The dataset may contain a large number of missing, incomplete, or unusable values
2. Success or failure may actually be influenced by factors not included in the dataset


# Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the dataset
data = pd.read_csv('data/startup_success_dataset.csv')

In [ ]:
# Display the first few rows of the dataset to understand its structure
data.head()

In [ ]:
# Display the shape of the dataset to better understand the number of samples and features
shape = data.shape
print(f"Rows:{shape[0]} \nColumns: {shape[1]}")

In [ ]:
# Display a summary of the dataset to understand its data types and identify any missing values
data.info()

In [ ]:
# Display basic statistical details to understand numerical distributions.
data.describe()

In [ ]:
# Display basic statistical details to understand categorical distributions.
data.describe(include = 'object')

In [ ]:
# Display a heatmap of all data with numeric values to visualize their correlations before the data is cleaned.

# Store the numeric data in a variable
data_numeric = data.select_dtypes(include = ['int64', 'float64'])

# Set the plot size, settings, and title.
plt.figure(figsize = (8, 6))
sns.heatmap(data_numeric.corr(), annot = True, cmap = 'coolwarm', vmin = -1, vmax = 1)
plt.xticks(rotation=45, ha='right')
plt.title('Heatmap of Correlated Numeric Data', y = 1.02)
plt.show()

The heatmap reveals that product_traction_users and revenue_million are highly correlated. This appears to be the strongest numeric correlation in the raw dataset.
However, all other numeric correlations appear near 0. This could mean highly synthetic data, or the relationships are nonlinear.

Based on this data, it may be beneficial to utilize _revenue_million_ and _product_traction_users_ in the initial model.
Additionally, a Decision Tree may find strong nonlinear patterns where the numeric correlation heatmap did not.

In [ ]:
# Display a histogram matrix to quickly identify outliers and skewed data
data_numeric.hist(figsize = (10, 7), bins = 20)
plt.tight_layout()
plt.suptitle('Histograms of Numeric Features', y = 1.02)
plt.show()

Several feeatures exhibit a right-skewed histogram. This likely indicates that scaling will be necessary later.

Additionally, features like _founder_experience_years_ and _team_size_ display unusually consistnt spikes and uniformity. This potentially means synthetic data.

_market_size_billion_ has a high quantitative value of startups operating in unusually large markets. This potentially indicates outliers.

In [ ]:
# Pair plot of all numeric features, visualizing their relationships and distributions.
sns.pairplot(data)
plt.suptitle('Pairplot of All Numeric Features', y = 1.02)
plt.show()

Based on the pairplot above, four features appear to have an identifiable correlation with the revenue factor, which was previously identified as a potential strong influence in startup outcome. These features include _funding_rounds_, _market_size_billion_, _product_traction_users_, and _burn_rate_million_.

In [ ]:

#Pair plot of the selected features
sns.pairplot(data[['revenue_million', 'burn_rate_million', 'product_traction_users', 'outcome']], hue = 'outcome')
plt.suptitle('Pairplot of Revenue, Burn Rate,Product Traction, and Outcome', y = 1.02)
plt.show()

Pair plots revealed a strong relationship between product traction and revenue across startup outcomes. Additionally, companies that reached IPO status generally exhibited the highest user traction and highest revenues. Contrary, failed startups typically had the lowest revenue and user count.

Furthermore, burn rate revealed a strong relationship with revenue and product traction. The data suggests that startups with higher burn rates had lower overall revenue and product traction.

Since revenue also demonstrates a strong relationship with startup outcomes, these features may indirectly contribute to predicting outcome classifications. Further analysis and modeling will help determine the strength and significance of these relationships.

Finally, the plot demonstrates clear outliers that may require cleaning before modeling.

In [ ]:
# Explore the target variable
data['outcome'].value_counts()

# Visualize outcome quantities
sns.countplot(data = data, x = 'outcome', hue = 'outcome')
plt.title('Outcome Quantities')
plt.show()

In [ ]:
# Visualize an outcome boxplot
sns.boxplot(data, x = 'outcome', y = 'revenue_million')
plt.title('Revenue by Outcome')
plt.show()

This boxplots confirms earlier suspicions of revenue potentially being a strong driver of outcome.
* Startups that reached an IPO appeared to generate the highest revenues. The median appears around ~$2.1M, while most IPO companies fall between ~$1.5 and ~$2.6M
* Failed companies typically produced the lowest revenues with the median falling around ~$400k. However, most failed startups fall between ~$250k and ~$700k
* Aquisitions typically happen between ~$500k and ~$1.5M in revenue. 

In [ ]:
# Explore the categorical values in the dataset
categorical_columns = data.select_dtypes(include = ['object']).columns
for column in categorical_columns:
    print(f"unique values in {column}: {data[column].unique()}")

In [ ]:
# Explore the investor_type feature where the value is 'none' to understand what the 'none' value represents in the context of the dataset.
data[data['investor_type'] == 'none'].head(10).reset_index(drop = True)

In [ ]:
# Inspect the investor_type feature to understand the outcome distribution across investor types.
(pd.crosstab(data['investor_type'], data['outcome'], normalize = 'index') * 100).plot(kind = 'bar', stacked = False)
plt.title('Outcome Distribution Across Investor Types', y = 1.05)
plt.ylabel('Percentage', labelpad = 20)
plt.xlabel('Investor Type', labelpad = 20)
plt.legend(title = 'outcome', bbox_to_anchor = (1.30, 1.025), loc = 'upper right')
plt.xticks(rotation = 360)
plt.show()

The outcome distribution is consistent across investor types. This feature may not have a high standalone contribution to outcome predictions. However, it may contribute predictive value when combined with other variables.

In [ ]:
sector_table = pd.crosstab(
    data['sector'],
    data['outcome'],
    normalize='index'
)

sns.heatmap(sector_table, annot=True)
plt.title('Outcome Distribution Across Sectors', y = 1.05)
plt.ylabel('Sector', labelpad = 20)
plt.xlabel('Outcome', labelpad = 20)
plt.show()

This heatmap suggests that business sector has a relatively weak standalone relationship with the outcome of a startup. Outcome distributions remain highly consistent across all sectors, and failure represents the majority. IPOs are consistently rare. While business sector may not have a standalone significance in outcome predictions, it may contribute predictive value when combined with other variables.

In [ ]:
founder_background_outcome = pd.crosstab(
    data['founder_background'],
    data['outcome'],
    normalize = 'index'
)

sns.heatmap(founder_background_outcome, annot = True)
plt.title('Outcome Distribution Across Founder Backgrounds', y = 1.05)
plt.ylabel('Founder Background', labelpad = 20)
plt.xlabel('Outcome', labelpad = 20)
plt.show()

This heatmap suggests that founder background has a relatively weak standalone relationship with the outcome of a startup. Outcome distributions remain highly consistent across all background types, and failure represents the majority. IPOs are consistently rare. While founder background may not have a standalone significance in outcome predictions, it may contribute predictive value when combined with other variables.

# Data Cleaning

In [ ]:
# Verify that all features have no missing values.
data.isnull().sum()

In [ ]:
# verify that there are no duplicate rows.
print(f"Number of duplicate rows: {data.duplicated().sum()}")

In [ ]:
# Copy the original dataset to a new variable to clean and prepare it for modeling.
model_data = data.copy()

In [ ]:
# Encode the target variable 'outcome' for modeling purposes.

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Fit the label_encoder to the 'outcome'
model_data['outcome'] = label_encoder.fit_transform(model_data['outcome'])

In [ ]:
# Save the mapping of the original outcome labels to their encoded values for future reference.
label_mapping = dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
))
print(label_mapping)

In [ ]:
# One-hot encode the remaining categorical features.
model_data = pd.get_dummies(
    model_data,
    columns = ['investor_type', 'sector', 'founder_background'],
    drop_first = True
)

# Modeling

In [ ]:
# Train a baseline logistic regression model to predict startup outcomes based on all features and lightly cleaned data.

# Separate the features (X) and the target variable (y | outcome).
X = model_data.drop('outcome', axis = 1)
y = model_data['outcome']

In [ ]:
# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
# Train the logistic regression model
model = LogisticRegression(random_state = 42, max_iter = 1000)
model.fit(X_train, y_train)

In [ ]:
# Make predictions on the test set
y_predictions = model.predict(X_test)

In [ ]:
# Evaluate the model's performance using accuracy_score
accuracy = accuracy_score(y_test, y_predictions)
print(f'Accuracy is: {accuracy: .2f}')

In [ ]:
# Explore the classification report to better understand performance across precision, recall, and f1-score for each class.
print(classification_report(y_test, y_predictions))

In [ ]:
# Visualize the results using a confusion matrix to better understand the quantitative performance across true positives/negatives, and false positives/negatives.
cmatrix = confusion_matrix(y_test, y_predictions)

sns.heatmap(cmatrix, annot=True, fmt = 'd', vmin = 0, cmap = 'Reds')

plt.title('Confusion Matrix', y = 1.05)
plt.xlabel('Predicted', labelpad = 20)
plt.ylabel('Actual', labelpad = 20)

plt.show()

In [ ]:
# Scale the data using StandardScaler() in hopes of improving the model performance. 
scaler = StandardScaler()

# scale the X datasets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model_scaled = LogisticRegression(random_state = 42, max_iter = 1000)
model_scaled.fit(X_train_scaled, y_train)

In [ ]:
# Make new predictions using the scaled data
y_prediction_scaled_data = model_scaled.predict(X_test_scaled)

# Evaluate the model's performance using accuracy_score
accuracy_scaled_data = accuracy_score(y_test, y_prediction_scaled_data)
print(f'Accuracy is: {accuracy_scaled_data: .2f}')

Scaling the data did improve the model by 15.625%.
Next, I'll explore whether log transformations on the heavily right-skewed data discovered in early EDA further improves the model. These include burn_rate_million, revenue_million, and market_size_billion.

In [ ]:
print(classification_report(y_test, y_prediction_scaled_data))